# Bài 20: Citations — trích dẫn nguồn
## Cho user biết câu trả lời dựa trên đâu

**Mục tiêu:**
- Giữ lại metadata (nguồn) mà ChromaDB vốn đã trả về
- Truyền nguồn qua state của graph
- Hiện nguồn dưới mỗi câu trả lời bằng `st.expander`

**Tại sao quan trọng nhất về độ tin cậy:** với tài chính, một câu trả lời không có nguồn thì không ai dám tin. Citations biến app từ "cái hộp đen phán bừa" thành "trợ lý có dẫn chứng" — đúng cách các product RAG thật (Perplexity, NotebookLM) tạo niềm tin. Nó cũng giúp bạn **debug retrieval**: thấy ngay app lấy nhầm slide thay vì báo cáo.

---
## Phần 1: Lý thuyết

### ChromaDB đã trả sẵn nguồn — ta chỉ chưa dùng

Mỗi lần `collection.query(...)` trả về một dict nhiều phần:

```python
results = collection.query(query_embeddings=[...], n_results=8, where={"symbol": symbol})

results["documents"][0]   # list text các chunk       ← lâu nay ta chỉ dùng cái này
results["metadatas"][0]   # list metadata mỗi chunk   ← {"symbol":..., "source":...}
results["distances"][0]   # list khoảng cách (càng nhỏ càng liên quan)
```

`documents[0]`, `metadatas[0]`, `distances[0]` — 3 list **song song, cùng độ dài**. Chunk thứ `i` có text `documents[0][i]` và nguồn `metadatas[0][i]`. Ghép lại bằng `zip()` là ra citation.

---

### Metadata có gì?

Nhớ lại: bài 13 lưu chunk với `{"symbol": "NVDA"}`, còn bài 19 (upload) lưu `{"symbol": ..., "source": tên_file}`.

→ Chunk NVDA/AMD cũ **không có** `"source"`. Nên khi đọc phải có fallback:
```python
source = meta.get("source", "báo cáo có sẵn")   # tránh KeyError
```

---

### Đường đi của citation qua hệ thống

```
rag/query (agents)   thu thập sources = [{symbol, source, snippet}, ...]
      ↓
state["sources"]     thêm 1 field mới vào AppState
      ↓
main.py              lưu sources kèm message, hiện trong st.expander
```

Điểm hay của kiến trúc module: chỉ đụng 3 file (`state.py`, `data_collection.py`, `main.py`), không đụng `analysis.py` hay `graph.py`.

---
## Phần 2: Ví dụ — xem metadata ChromaDB trả về

Chạy ô dưới để thấy `documents`, `metadatas`, `distances` đi song song thế nào.

In [1]:
import chromadb
from sentence_transformers import SentenceTransformer

embed = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="../data/chroma_db")
col = client.get_collection("multi_company_reports")

q = embed.encode("quarterly revenue").tolist()
results = col.query(query_embeddings=[q], n_results=3, where={"symbol": "NVDA"})

docs   = results["documents"][0]
metas  = results["metadatas"][0]
dists  = results["distances"][0]

# 3 list song song — ghép bằng zip
for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists)):
    source = meta.get("source", "báo cáo có sẵn")   # fallback vì chunk cũ không có 'source'
    print(f"--- Chunk {i} | symbol={meta['symbol']} | source={source} | distance={dist:.3f} ---")
    print(doc[:150], "...\n")

d:\Code\AI\Multi-Agent Research Assistant project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4214.72it/s]


--- Chunk 0 | symbol=NVDA | source=báo cáo có sẵn | distance=0.942 ---
period $ 2,078 $ 1,549 (1) Deferred revenue additions includes $6.2 billion and $157 million of customer advances for the first quarter of fiscal year ...

--- Chunk 1 | symbol=NVDA | source=NVDA-F1Q26-10-Q.pdf | distance=0.942 ---
period $ 2,078 $ 1,549 (1) Deferred revenue additions includes $6.2 billion and $157 million of customer advances for the first quarter of fiscal year ...

--- Chunk 2 | symbol=NVDA | source=báo cáo có sẵn | distance=0.999 ---
segment. Indirect Customers – Indirect customer revenue is an estimation based upon multiple factors including customer purchase order information, pr ...



Thấy rõ: mỗi chunk đi kèm `symbol`, `source`, `distance`. Đó là tất cả nguyên liệu để làm citation.

---
## Phần 3: Bài tập

### Bước 1: Thêm field `sources` vào `app/core/state.py`

```python
class AppState(TypedDict):
    messages: Annotated[list[dict], add]
    symbols: list[str]
    company_data: dict
    sources: list[dict]        # ← THÊM: [{"symbol":..., "source":..., "snippet":...}, ...]
```

---

### Bước 2: `collect_for_symbol()` trả thêm sources — `app/agents/data_collection.py`

Hiện tại hàm chỉ `return` một chuỗi text. Giờ cho nó trả về **cả text lẫn danh sách nguồn**.

Sau dòng lấy `report_chunks`, thêm phần dựng sources (điền TODO):

```python
    report_chunks = results["documents"][0] if results["documents"] else []
    metadatas     = results["metadatas"][0] if results["metadatas"] else []

    # TODO: dựng list sources bằng zip(report_chunks, metadatas)
    #   mỗi phần tử là dict: {
    #       "symbol":  symbol,
    #       "source":  meta.get("source", "báo cáo có sẵn"),   # nhớ fallback
    #       "snippet": chunk[:200],                            # 200 ký tự đầu để user xem
    #   }
    sources = [ ... ]
```

Và đổi dòng `return` cuối — trả về **tuple** `(text, sources)`:
```python
    return (
        f"=== {symbol} ===\n"
        f"[Tài chính]\n{financial_snapshot}\n"
        f"[Báo cáo]\n{report_text}"
    ), sources
```

---

### Bước 3: `data_collection_node()` gom sources — cùng file

Vì `collect_for_symbol` giờ trả 2 giá trị, phải sửa vòng lặp cho khớp (điền TODO):

```python
def data_collection_node(state: AppState) -> dict:
    logger.info(f"[data_collection] Bắt đầu thu thập cho {len(state['symbols'])} symbols: {state['symbols']}")
    question = state["messages"][-1]["content"]
    company_data = {}
    all_sources = []
    for symbol in state["symbols"]:
        # TODO: nhận 2 giá trị trả về
        #   text, sources = collect_for_symbol(symbol, question)
        #   company_data[symbol] = text
        #   all_sources.extend(sources)      # gộp nguồn của mọi công ty
        pass
    # TODO: return cả company_data lẫn sources
    return {"company_data": company_data, "sources": all_sources}
```

---

### Bước 4: Hiện citations trong `app/main.py`

**4a.** Khi lưu câu trả lời vào session_state, đính kèm sources:
```python
    answer = result["messages"][-1]["content"]
    st.session_state.messages.append({
        "role": "assistant",
        "content": answer,
        "sources": result.get("sources", []),     # ← đính kèm
    })
```

**4b.** Trong vòng lặp render lịch sử, thêm expander khi message có sources:
```python
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"].replace("$", r"\$"))
        # TODO: nếu msg.get("sources") có dữ liệu → hiện st.expander
        #   with st.expander(f"📎 Nguồn tham khảo ({len(...)} đoạn)"):
        #       for s in msg["sources"]:
        #           st.markdown(f"**[{s['symbol']}]** `{s['source']}`")
        #           st.caption(s["snippet"].replace("$", r"\$") + "...")
```

> Lưu ý: lưu sources **trong message dict** (không phải biến rời) để khi Streamlit rerun, mỗi câu trả lời cũ vẫn giữ nguồn của nó.

**4c.** Đừng quên `initial_state` cũng cần key `sources`:
```python
        initial_state: AppState = {
            "symbols": symbols,
            "messages": [{"role": "user", "content": question}],
            "company_data": {},
            "sources": [],
        }
```

---

### Bước 5: Test

`streamlit run app/main.py` → hỏi "So sánh NVDA và TSLA". Dưới câu trả lời phải có mục **📎 Nguồn tham khảo** bung ra được, liệt kê các chunk đã dùng kèm tên file.

**Checklist:**
- [ ] Expander hiện đúng số đoạn
- [ ] Chunk TSLA hiện `source` = tên file bạn upload; chunk NVDA hiện "báo cáo có sẵn"
- [ ] Hỏi câu thứ 2 → câu trả lời cũ vẫn giữ nguồn riêng (không bị mất khi rerun)

**Bonus quan sát:** nhìn snippet NVDA — bạn sẽ thấy ngay retrieval đang lấy chunk nào. Nếu toàn slide mà thiếu bảng doanh thu → đó là lý do câu trả lời thiếu số liệu. Citations vừa tạo niềm tin, vừa là công cụ debug.